# Финальный eval: все методы CPD на 50 рядах x 4 сценария

**Перед запуском**: добавьте dataset с файлами весов (.pt, .cbm) через
Add Data → Your Datasets. Ноутбук автоматически найдёт их в `/kaggle/input/`.

Сценарии:
- D=0.5, white noise (мало CP, лёгкий)
- D=1.0, white noise (много CP, как в ноутбуке)
- D=1.0, pink noise (цветной шум)
- D=1.5, white noise (очень много CP)

Результаты сохраняются в `/kaggle/working/eval_multi_{scenario}/`.

In [ ]:
!pip install pyhomogeneity -q


In [ ]:
import math, time, csv, gc
from dataclasses import dataclass
from math import erf, sqrt
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from scipy import stats
from scipy.ndimage import binary_dilation
from scipy.stats import f as f_dist
from sklearn.metrics import roc_auc_score, average_precision_score
import pyhomogeneity as hg

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('torch:', torch.__version__)

N_SERIES = 50
TEST_SEED_START = 1000
MARGIN = 25
NAN_MAE = 9999.0

SCENARIOS = [
    {'D': 0.5, 'noise_type': 'white', 'tag': 'D05_white'},
    {'D': 1.0, 'noise_type': 'white', 'tag': 'D10_white'},
    {'D': 1.0, 'noise_type': 'pink',  'tag': 'D10_pink'},
    {'D': 1.5, 'noise_type': 'white', 'tag': 'D15_white'},
]

BASE_OUTPUT = Path('/kaggle/working')

def _find_model(filename):
    for f in Path('/kaggle/input').rglob(filename):
        return str(f)
    print('WARNING: not found:', filename)
    return None

print(f'config OK, {len(SCENARIOS)} scenarios, margin={MARGIN}')

In [ ]:
def _noise_psd(N, psd):
    X_white = np.fft.rfft(np.random.randn(N))
    S = psd(np.fft.rfftfreq(N))
    S = S / np.sqrt(np.mean(S**2))
    return np.fft.irfft(X_white * S)

def white_noise(N):    return _noise_psd(N, lambda f: np.ones_like(f))
def pink_noise(N):     return _noise_psd(N, lambda f: 1 / np.where(f == 0, float('inf'), np.sqrt(f)))
def brownian_noise(N): return _noise_psd(N, lambda f: 1 / np.where(f == 0, float('inf'), f))
def violet_noise(N):   return _noise_psd(N, lambda f: f)
def blue_noise(N):     return _noise_psd(N, lambda f: np.sqrt(f))

_NOISE_FN = {'white': white_noise, 'pink': pink_noise,
             'brownian': brownian_noise, 'violet': violet_noise, 'blue': blue_noise}

def get_noise(N, noise_type='white'):
    return _NOISE_FN[noise_type](N)

def generate_sde_trajectory(length, dt=1.0, D=0.5, noise_type='white', seed=42):
    np.random.seed(seed)
    errors = get_noise(length, noise_type)
    errors -= np.mean(errors)
    std = np.std(errors)
    if std > 1e-12:
        errors /= std
    x_t = math.pi / 1e6
    data = [x_t]
    for i in range(length):
        x_t = x_t + math.sin(x_t) * dt + math.sqrt(D) * errors[i]
        data.append(x_t)
    return np.array(data)

def compute_levels(x_values):
    n = len(x_values)
    pi_list = np.zeros(n)
    level_list = np.zeros(n, dtype=int)
    for i in range(1, n):
        temp = x_values[i] / math.pi
        pi_list[i] = temp
        prev_level = level_list[i - 1]
        prev_pi = pi_list[i - 1]
        if (prev_pi > 0 and temp < 0) or (prev_pi < 0 and temp > 0):
            if abs(temp - prev_level) >= 1:
                if temp > 0:
                    level_list[i] = prev_level + math.floor(abs(temp - prev_level))
                else:
                    level_list[i] = prev_level - math.floor(abs(temp - prev_level))
            else:
                level_list[i] = prev_level
        else:
            diff = abs(temp) - abs(prev_level)
            if diff >= 1:
                if temp > 0:
                    level_list[i] = prev_level + math.floor(diff)
                else:
                    level_list[i] = prev_level - math.floor(diff)
            elif diff <= -1:
                if temp > 0:
                    level_list[i] = prev_level - math.floor(abs(diff))
                else:
                    level_list[i] = prev_level + math.floor(abs(diff))
            else:
                level_list[i] = prev_level
    cp_labels = np.zeros(n, dtype=int)
    for i in range(1, n):
        if level_list[i] != level_list[i - 1]:
            cp_labels[i] = 1
    return level_list, cp_labels

def generate_dataset(length, dt=1.0, D=0.5, noise_type='white', seed=42):
    x = generate_sde_trajectory(length, dt, D, noise_type, seed)
    _, cp = compute_levels(x)
    return x, cp

def generate_multi_dataset(n_seq, seq_length, dt=1.0, D_range=(0.3, 2.0),
                           noise_types=None, seed=42, dt_values=None, t4_fraction=0.0):
    if noise_types is None:
        noise_types = ['white']
    rng = np.random.RandomState(seed)
    all_x, all_cp = [], []
    for _ in range(n_seq):
        D = rng.uniform(D_range[0], D_range[1])
        seq_seed = rng.randint(0, 100000)
        noise_type = noise_types[rng.randint(0, len(noise_types))]
        chosen_dt = dt_values[rng.randint(0, len(dt_values))] if dt_values else dt
        x, cp = generate_dataset(seq_length, chosen_dt, D, noise_type, seq_seed)
        all_x.append(x)
        all_cp.append(cp)
    return np.concatenate(all_x), np.concatenate(all_cp)

print('data_utils OK')


In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, x_values, cp_labels, window_size=100, margin=5):
        self.window_size = window_size
        half = window_size // 2
        windows, labels = [], []
        for i in range(half, len(x_values) - half):
            window = x_values[i - half:i + half]
            left = max(0, i - margin)
            right = min(len(cp_labels), i + margin + 1)
            label = 1 if np.any(cp_labels[left:right] == 1) else 0
            windows.append(window)
            labels.append(label)
        self.windows = np.array(windows, dtype=np.float32)
        self.labels = np.array(labels, dtype=np.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        w = self.windows[idx]
        std = w.std()
        w = (w - w.mean()) / std if std > 1e-8 else w - w.mean()
        return torch.tensor(w, dtype=torch.float32).unsqueeze(-1), torch.tensor(self.labels[idx], dtype=torch.float32)


def evaluate_batch(model, loader, device='cpu', threshold=0.5):
    model.eval()
    tp = fp = fn = 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds = (torch.sigmoid(model(x_batch)) >= threshold).float()
            tp += ((preds == 1) & (y_batch == 1)).sum().item()
            fp += ((preds == 1) & (y_batch == 0)).sum().item()
            fn += ((preds == 0) & (y_batch == 1)).sum().item()
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    return f1, precision, recall


def predict_on_series(model, x_values, window_size=100, device='cpu', threshold=0.5):
    model.eval()
    half = window_size // 2
    scores = np.full(len(x_values), np.nan)
    windows, indices = [], []
    for i in range(half, len(x_values) - half):
        windows.append(x_values[i - half:i + half])
        indices.append(i)
    if not windows:
        return scores, np.zeros(len(x_values), dtype=int)
    windows_arr = np.array(windows, dtype=np.float32)
    means = windows_arr.mean(axis=1, keepdims=True)
    stds = windows_arr.std(axis=1, keepdims=True)
    stds[stds < 1e-8] = 1.0
    windows_arr = (windows_arr - means) / stds
    windows_tensor = torch.tensor(windows_arr).unsqueeze(-1).to(device)
    with torch.no_grad():
        all_probs = []
        for start in range(0, len(windows_tensor), 1024):
            logits = model(windows_tensor[start:start + 1024])
            all_probs.append(torch.sigmoid(logits).cpu().numpy())
        all_probs = np.concatenate(all_probs)
    for idx, prob in zip(indices, all_probs):
        scores[idx] = prob
    preds = np.zeros(len(x_values), dtype=int)
    preds[~np.isnan(scores)] = (scores[~np.isnan(scores)] >= threshold).astype(int)
    return scores, preds


def nms_predictions(scores, threshold=0.5, min_distance=50):
    preds = np.zeros(len(scores), dtype=int)
    candidates = np.where(scores >= threshold)[0]
    if len(candidates) == 0:
        return preds
    order = candidates[np.argsort(-scores[candidates])]
    suppressed = set()
    for idx in order:
        if idx in suppressed:
            continue
        preds[idx] = 1
        for j in range(max(0, idx - min_distance), min(len(scores), idx + min_distance + 1)):
            if j != idx:
                suppressed.add(j)
    return preds


def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3,
                pos_weight=None, device='cpu', patience=15, weight_decay=1e-4):
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight], device=device) if pos_weight is not None else None
    ) if pos_weight else nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    model.to(device)
    best_f1, best_state, no_improve = 0.0, None, 0
    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        total_loss, n_batches = 0, 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1
        val_f1, val_prec, val_rec = evaluate_batch(model, val_loader, device)
        print(f'Epoch {epoch+1:3d}/{epochs} | Loss: {total_loss/max(n_batches,1):.4f} | Val F1: {val_f1:.3f} | {time.time()-t0:.0f}s', flush=True)
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break
    if best_state:
        model.load_state_dict(best_state)
    return model.to(device)

print('ML utilities OK')


In [ ]:
class LSTMChangePointDetector(nn.Module):
    def __init__(self, input_size=1, hidden_size=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
                            dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class GRUChangePointDetector(nn.Module):
    def __init__(self, input_size=1, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size=input_size, hidden_size=hidden_size, num_layers=num_layers,
                          dropout=dropout if num_layers > 1 else 0.0, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=500):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TransformerChangePointDetector(nn.Module):
    def __init__(self, input_size=1, d_model=256, nhead=8, num_layers=6, dim_feedforward=512, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead,
                                                   dim_feedforward=dim_feedforward,
                                                   dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, 1)
    def forward(self, x):
        x = self.pos_encoding(self.input_proj(x))
        return self.fc(self.encoder(x).mean(dim=1)).squeeze(-1)

print('models OK')


In [ ]:
# ---- pyhomogeneity-based methods ----

def _hg_meta(kwargs, result, seed):
    return {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', ''),
            'p_value': float(result.p), 'seed': seed}

def _hg_cp(result, alpha=0.05):
    if result.p > alpha or result.cp is None:
        return []
    return [int(result.cp)]

def _sk(series):
    return np.cumsum(series - np.mean(series))

def pettitt(series, seed, alpha=0.05, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    r = hg.pettitt_test(series)
    return _hg_cp(r, alpha), None, _hg_meta(kwargs, r, seed)

def snht(series, seed, alpha=0.05, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    r = hg.snht_test(series)
    return _hg_cp(r, alpha), None, _hg_meta(kwargs, r, seed)

def buishand_q(series, seed, alpha=0.05, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    r = hg.buishand_q_test(series)
    m = _hg_meta(kwargs, r, seed)
    return _hg_cp(r, alpha), _sk(series), m

def buishand_range(series, seed, alpha=0.05, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    r = hg.buishand_range_test(series)
    return _hg_cp(r, alpha), _sk(series), _hg_meta(kwargs, r, seed)

def buishand_likelihood_ratio(series, seed, alpha=0.05, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    r = hg.buishand_likelihood_ratio_test(series)
    return _hg_cp(r, alpha), _sk(series), _hg_meta(kwargs, r, seed)

def buishand_u(series, seed, alpha=0.05, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    r = hg.buishand_u_test(series)
    return _hg_cp(r, alpha), _sk(series), _hg_meta(kwargs, r, seed)


# ---- CUSUM ----

class CusumMeanDetector:
    def __init__(self, k=0.5, h=5.0, warmup=30):
        self._k, self._h, self._warmup = k, h, int(warmup)
        self._buf, self._mu, self._sigma = [], 0.0, 1.0
        self._sp, self._sm, self._ready = 0.0, 0.0, False
    def predict_next(self, y):
        if not self._ready:
            self._buf.append(float(y))
            if len(self._buf) >= self._warmup:
                arr = np.array(self._buf)
                self._mu = float(np.mean(arr))
                self._sigma = max(float(np.std(arr)), 1e-12)
                self._ready = True
            return 0.0, False
        slack = self._k * self._sigma
        threshold = self._h * self._sigma
        self._sp = max(0.0, self._sp + (float(y) - self._mu - slack))
        self._sm = max(0.0, self._sm - (float(y) - self._mu + slack))
        confidence = max(self._sp, self._sm) / threshold
        is_cp = confidence >= 1.0
        if is_cp:
            self._sp = self._sm = 0.0
        return float(confidence), is_cp

def run_cusum(series, seed, t_warmup=30, k=0.5, h=5.0, p_limit=0.01, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    n = len(series)
    detector = CusumMeanDetector(k=k, h=h, warmup=t_warmup)
    scores = np.zeros(n)
    cps, skip_until = [], 0
    for t in range(n):
        conf, is_cp = detector.predict_next(series[t])
        scores[t] = conf
        if is_cp and t >= skip_until:
            cps.append(t)
            skip_until = t + t_warmup
    meta = {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', ''), 'seed': int(seed)}
    return cps, scores, meta


# ---- Chow ----

from sklearn.linear_model import LinearRegression

def _rss(x, y):
    model = LinearRegression().fit(x.reshape(-1, 1), y)
    return float(((y - model.predict(x.reshape(-1, 1))) ** 2).sum())

def _chow_f_at(series, t0):
    n = len(series)
    x = np.arange(n, dtype=float)
    k = 2
    rss_pooled = _rss(x, series)
    rss1 = _rss(x[:t0], series[:t0])
    rss2 = _rss(x[t0:], series[t0:])
    num = (rss_pooled - (rss1 + rss2)) / k
    den = (rss1 + rss2) / (n - 2 * k)
    f_stat = num / den if den > 0 else 0.0
    p_val = float(1 - f_dist.cdf(f_stat, dfn=k, dfd=(n - 2 * k)))
    return f_stat, p_val

def chow(series, seed, t0=None, alpha=0.05, min_segment=10, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    n = len(series)
    if t0 is not None:
        f_stat, p_val = _chow_f_at(series, int(t0))
        scores, best_t0 = None, int(t0)
    else:
        scores = np.zeros(n)
        for c in range(min_segment, n - min_segment):
            f, _ = _chow_f_at(series, c)
            scores[c] = f
        best_t0 = int(np.argmax(scores))
        f_stat, p_val = _chow_f_at(series, best_t0)
    cps = [best_t0] if p_val < alpha else []
    meta = {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', ''),
            'f_statistic': f_stat, 'p_value': p_val, 'seed': seed}
    return cps, scores, meta

print('stat methods 1 OK')


In [ ]:
# ---- MDL ----

def quantize_fixed(x, lo, hi, b=6):
    card = 1 << int(b)
    if hi <= lo:
        return np.zeros_like(x, dtype=np.int16)
    y = (np.asarray(x, dtype=float) - lo) / (hi - lo) * (card - 1)
    return np.clip(np.floor(y + 1e-12).astype(np.int32), 0, card - 1).astype(np.int16)

def entropy_q(q, card):
    q = np.asarray(q)
    if q.size == 0: return 0.0
    counts = np.bincount(q.astype(np.int64), minlength=int(card))
    counts = counts[counts > 0]
    p = counts / q.size
    return float(-(p * np.log2(p)).sum())

def dl_q(q, card):
    return float(np.asarray(q).size * entropy_q(q, card))

def mdl_score_window(y, cp, win, b=6):
    y = np.asarray(y, dtype=float).reshape(-1)
    l0, r1 = cp - win + 1, cp + 1 + win
    if l0 < 0 or r1 > y.size: return None
    full = y[l0:r1]
    lo, hi = float(full.min()), float(full.max())
    card = 1 << int(b)
    q_full = quantize_fixed(full, lo, hi, b=b)
    score = dl_q(q_full, card) - dl_q(q_full[:win], card) - dl_q(q_full[win:], card)
    return float(max(0.0, score))

def mdl_candidates(y, win=120, b=6, stride=2, min_cp_gap=120, top_k=25):
    y = np.asarray(y, dtype=float).reshape(-1)
    n = y.size
    if n < 2 * win + 2: return []
    idx = list(range(win - 1, n - win - 1, stride))
    scores = np.zeros(len(idx))
    for k, cp in enumerate(idx):
        sc = mdl_score_window(y, cp, win, b)
        scores[k] = sc if sc is not None else 0.0
    peaks = [i for i in range(1, scores.size - 1)
             if scores[i] > scores[i - 1] and scores[i] >= scores[i + 1] and scores[i] > 0]
    if not peaks:
        peaks = list(np.argsort(-scores)[:max(1, min(top_k, scores.size))])
    peaks = sorted(peaks, key=lambda i: -scores[i])
    chosen = []
    for i in peaks:
        cp = int(idx[i])
        if all(abs(cp - c) >= min_cp_gap for c in chosen):
            chosen.append(cp)
            if len(chosen) >= top_k: break
    chosen.sort()
    return chosen

@dataclass
class _Seg:
    t0: int; t1: int; n: int; st: float; st2: float; sy: float; sty: float; sy2: float
    @staticmethod
    def from_arr(t, y, i0, i1):
        tt, yy = t[i0:i1], y[i0:i1]
        return _Seg(int(tt[0]), int(tt[-1]), int(yy.size), float(tt.sum()), float((tt*tt).sum()),
                    float(yy.sum()), float((tt*yy).sum()), float((yy*yy).sum()))
    def merge(self, o):
        return _Seg(self.t0, o.t1, self.n+o.n, self.st+o.st, self.st2+o.st2,
                    self.sy+o.sy, self.sty+o.sty, self.sy2+o.sy2)
    def fit_ab(self):
        d = self.n*self.st2 - self.st*self.st
        if d == 0: return self.sy/self.n, 0.0
        b = (self.n*self.sty - self.st*self.sy) / d
        return (self.sy - b*self.st)/self.n, b
    def sse_linear(self):
        a, b = self.fit_ab()
        sse = self.sy2 - 2*a*self.sy - 2*b*self.sty + self.n*a*a + 2*a*b*self.st + b*b*self.st2
        return float(max(0.0, sse))
    def sse_const(self):
        if self.n <= 0: return 0.0
        mu = self.sy/self.n
        return float(max(0.0, self.sy2 - 2*mu*self.sy + self.n*mu*mu))

def _chow_win(y, cp, win, model='linear'):
    y = np.asarray(y, dtype=float).reshape(-1)
    l0, l1, r0, r1 = cp-win+1, cp+1, cp+1, cp+1+win
    if l0 < 0 or r1 > y.size: return None
    t = np.arange(y.size, dtype=int)
    left, right = _Seg.from_arr(t, y, l0, l1), _Seg.from_arr(t, y, r0, r1)
    merged = left.merge(right)
    k = 1 if model == 'const' else 2
    sse_l = left.sse_const() if model=='const' else left.sse_linear()
    sse_r = right.sse_const() if model=='const' else right.sse_linear()
    sse_m = merged.sse_const() if model=='const' else merged.sse_linear()
    df2 = left.n + right.n - 2*k
    if df2 <= 0: return 0.0, 1.0, 0.0, k, df2
    den = (sse_l + sse_r) / df2
    if den <= 0: return 0.0, 1.0, 0.0, k, df2
    F = max(0.0, (sse_m - (sse_l + sse_r)) / k / den)
    p_val = float(f_dist.sf(F, k, df2))
    return 1.0 - p_val, p_val, F, k, df2

def _detect_cps(y, cps, win=120, model='linear', alpha_fwer=0.05, min_cp_gap=100):
    cps = sorted(int(x) for x in cps)
    m = max(1, len(cps))
    thr = 1.0 - alpha_fwer / m
    events, last_cp = [], None
    for cp in cps:
        if last_cp is not None and abs(cp - last_cp) < min_cp_gap: continue
        out = _chow_win(y, cp, win, model)
        if out is None: continue
        prob = out[0]
        if prob >= thr:
            events.append(cp)
            last_cp = cp
    return thr, events

def mdl(series, seed, win=120, b=6, stride=2, min_cp_gap_cand=120, top_k=25,
        model='const', alpha_fwer=0.05, min_cp_gap=100, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    cps = mdl_candidates(series, win=win, b=b, stride=stride, min_cp_gap=min_cp_gap_cand, top_k=top_k)
    thr, events = _detect_cps(series, cps, win=win, model=model, alpha_fwer=alpha_fwer, min_cp_gap=min_cp_gap)
    meta = {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', ''), 'seed': seed}
    return events, None, meta


# ---- SWAB ----

def _initial_segs(t, y, min_len):
    n = y.size
    cuts = []
    i = 0
    while i < n:
        j = min(i + min_len, n)
        cuts.append((i, j))
        i = j
    if len(cuts) >= 2 and (cuts[-1][1] - cuts[-1][0]) < min_len:
        a0, a1 = cuts[-2]; b0, b1 = cuts[-1]
        cuts = cuts[:-2] + [(a0, b1)]
    return [_Seg.from_arr(t, y, i0, i1) for i0, i1 in cuts]

def _bottom_up(t, y, min_len, target=6):
    segs = _initial_segs(t, y, min_len)
    while len(segs) > max(1, target):
        best_i, best_cost, best_merged = -1, None, None
        for i in range(len(segs) - 1):
            merged = segs[i].merge(segs[i+1])
            cost = merged.sse_linear() - segs[i].sse_linear() - segs[i+1].sse_linear()
            if best_cost is None or cost < best_cost:
                best_cost, best_i, best_merged = cost, i, merged
        if best_i < 0: break
        segs = segs[:best_i] + [best_merged] + segs[best_i+2:]
    return segs

def swab_candidates(y, w=400, min_len=50, target=6):
    y = np.asarray(y, dtype=float).reshape(-1)
    n, t = y.size, np.arange(y.size, dtype=int)
    cps, left = [], 0
    while left < n:
        right = min(left + w, n)
        yb = y[left:right]
        if yb.size < min_len: break
        segs = _bottom_up(t[left:right], yb, min_len, target)
        if not segs: break
        cp = int(segs[0].t1)
        if cp < n - 1: cps.append(cp)
        left = cp + 1
    return cps

def swab(series, seed, w=400, min_len=50, target_segments=6, win=200,
         model='const', alpha_fwer=0.05, min_cp_gap=100, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    cps = swab_candidates(series, w=w, min_len=min_len, target=target_segments)
    _, events = _detect_cps(series, cps, win=win, model=model, alpha_fwer=alpha_fwer, min_cp_gap=min_cp_gap)
    meta = {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', ''), 'seed': seed}
    return events, None, meta


# ---- GP ----

def _norm_cdf(z):
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))

def _rbf(d2, sf2, ell):
    if ell <= 0: return sf2 * (d2 == 0).astype(float)
    return sf2 * np.exp(-0.5 * d2 / (ell * ell))

def _gp_cache(m, sf2, sn2, ell):
    idx = np.arange(m, dtype=float)
    diff = idx[:, None] - idx[None, :]
    K = _rbf(diff*diff, sf2, ell) + (sn2 + 1e-12) * np.eye(m)
    jitter = 1e-10 * (sf2 + sn2 + 1.0)
    for _ in range(6):
        try:
            L = np.linalg.cholesky(K + jitter * np.eye(m))
            break
        except np.linalg.LinAlgError:
            jitter *= 10
    else:
        return None
    kstar_d2 = (idx - float(m)) ** 2
    k_star = _rbf(kstar_d2, sf2, ell)
    v = np.linalg.solve(L, k_star.reshape(-1, 1))
    base_var = max(1e-12, float(sf2 + sn2) - float((v*v).sum()))
    return L, k_star, base_var

def _gp_predict(L, k_star, base_var, y_train):
    y_train = np.asarray(y_train).reshape(-1, 1)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y_train))
    mu = float(k_star @ alpha.reshape(-1))
    return mu, base_var

def gp_candidates(y, train_win=180, alpha=1e-3, min_cp_gap=60, noise_ratio=0.12, ell_ratio=0.35):
    y = np.asarray(y, dtype=float).reshape(-1)
    n, m = y.size, int(train_win)
    if n <= m + 2: return []
    sigma = float(np.std(y[:max(m, 5)]))
    if sigma <= 1e-12: sigma = float(np.std(y))
    if sigma <= 1e-12: return []
    cache = _gp_cache(m, sigma**2, (noise_ratio*sigma)**2, max(1.0, ell_ratio*m))
    if cache is None: return []
    L, k_star, base_var = cache
    cand = []
    for t in range(m, n):
        mu, var = _gp_predict(L, k_star, base_var, y[t-m:t])
        z = abs(y[t] - mu) / sqrt(var)
        p = 2.0 * (1.0 - _norm_cdf(z))
        if p < float(alpha): cand.append((int(t-1), p))
    if not cand: return []
    cand.sort(key=lambda x: x[1])
    chosen = []
    for cp, _ in cand:
        if all(abs(cp - c) >= min_cp_gap for c in chosen): chosen.append(cp)
    chosen.sort()
    return chosen

def gp(series, seed, train_win=180, alpha_cand=1e-3, min_cp_gap_cand=60,
       win=200, model='const', alpha_fwer=0.05, min_cp_gap=200, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    cps = gp_candidates(series, train_win=train_win, alpha=alpha_cand, min_cp_gap=min_cp_gap_cand)
    _, events = _detect_cps(series, cps, win=win, model=model, alpha_fwer=alpha_fwer, min_cp_gap=min_cp_gap)
    meta = {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', ''), 'seed': seed}
    return events, None, meta


# ---- Nyblom ----

def nyblom_stat(y, X=None):
    y = np.asarray(y, dtype=float).reshape(-1)
    n = y.shape[0]
    if X is None: X = np.ones((n, 1))
    else: X = np.asarray(X, dtype=float).reshape(n, -1) if np.asarray(X).ndim == 1 else np.asarray(X, dtype=float)
    beta_hat = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta_hat
    sigma2 = float(resid @ resid) / n
    V = X.T @ X / n
    k = X.shape[1]
    S = np.zeros(k)
    q = np.empty(n)
    try:
        R = np.linalg.cholesky(V)
        use_chol = True
    except Exception:
        use_chol = False
    for t in range(n):
        S = S + X[t] * resid[t]
        z = np.linalg.solve(R, S) if use_chol else np.linalg.lstsq(V, S, rcond=None)[0]
        q[t] = float(z @ z) if use_chol else float(S @ z)
    L = float(np.sum(q)) / (n * n * sigma2) if sigma2 > 0 else float('nan')
    return L, q, beta_hat, resid

def nyblom_cv(L, k, alpha=0.05, m=2000, grid=2048, seed=0):
    rng = np.random.default_rng(seed)
    dt = 1.0 / grid
    sims = np.empty(m)
    for i in range(m):
        inc = rng.normal(0.0, sqrt(dt), size=(grid, k))
        W = np.cumsum(inc, axis=0)
        lam = (np.arange(1, grid+1, dtype=float) / grid).reshape(-1, 1)
        B = W - lam * W[-1].reshape(1, -1)
        sims[i] = float(np.sum(B*B)) * dt
    cv = float(np.quantile(sims, 1.0 - alpha))
    p_value = float((np.sum(sims >= L) + 1.0) / (m + 1.0))
    return cv, p_value

def nyblom(series, seed, X=None, alpha=0.05, m=2000, grid=2048, **kwargs):
    series = np.asarray(series, dtype=float).reshape(-1)
    L, q, beta_hat, resid = nyblom_stat(series, X)
    k = 1 if X is None else (np.asarray(X).reshape(len(series), -1).shape[1])
    cv, p_value = nyblom_cv(L, k, alpha=alpha, m=m, grid=grid, seed=seed)
    reject = bool(L > cv)
    cp_index = int(np.argmax(q))
    cps = [cp_index] if reject else []
    meta = {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', ''),
            'p_value': p_value, 'stat': L, 'reject': reject, 'seed': seed}
    return cps, q, meta

print('stat methods 2 OK')


In [ ]:
# ---- CatBoost CPD ----

def _catboost_features_batch(windows):
    N, W = windows.shape
    half = W // 2
    diff = np.diff(windows, axis=1)
    t = np.arange(W, dtype=np.float64)
    t_c = t - t.mean()
    t_var = (t_c ** 2).sum()
    w_mean = windows.mean(axis=1, keepdims=True)
    slope = ((windows - w_mean) * t_c).sum(axis=1) / t_var
    x_ac, y_ac = windows[:, :-1], windows[:, 1:]
    xm, ym = x_ac.mean(axis=1, keepdims=True), y_ac.mean(axis=1, keepdims=True)
    num = ((x_ac - xm) * (y_ac - ym)).sum(axis=1)
    den = np.sqrt(((x_ac - xm)**2).sum(axis=1) * ((y_ac - ym)**2).sum(axis=1))
    autocorr = np.where(den > 1e-8, num / den, 0.0)
    features = np.column_stack([
        windows.mean(axis=1), windows.std(axis=1), windows.min(axis=1), windows.max(axis=1),
        windows.max(axis=1) - windows.min(axis=1),
        stats.skew(windows, axis=1), stats.kurtosis(windows, axis=1),
        windows[:, :half].mean(axis=1) - windows[:, half:].mean(axis=1),
        windows[:, :half].std(axis=1) - windows[:, half:].std(axis=1),
        slope, autocorr,
        np.abs(diff).mean(axis=1), diff.std(axis=1),
        np.median(windows, axis=1),
    ]).astype(np.float32)
    return np.where(np.isfinite(features), features, 0.0)

def _make_norm_windows(x_values, window_size):
    x = np.asarray(x_values, dtype=np.float64)
    half = window_size // 2
    indices = np.arange(half, len(x_values) - half)
    n = len(indices)
    windows_raw = np.lib.stride_tricks.sliding_window_view(x, window_size)[:n]
    means = windows_raw.mean(axis=1, keepdims=True)
    stds = windows_raw.std(axis=1, keepdims=True)
    stds = np.where(stds < 1e-8, 1.0, stds)
    return ((windows_raw - means) / stds).astype(np.float32), indices

def predict_catboost(model, x_values, window_size, threshold=0.5):
    scores = np.full(len(x_values), np.nan)
    windows_norm, indices = _make_norm_windows(x_values, window_size)
    if len(windows_norm) == 0:
        return scores, np.zeros(len(x_values), dtype=int)
    X = _catboost_features_batch(windows_norm)
    probs = model.predict_proba(X)[:, 1]
    scores[indices] = probs
    preds = np.zeros(len(x_values), dtype=int)
    valid = ~np.isnan(scores)
    preds[valid] = (scores[valid] >= threshold).astype(int)
    return scores, preds


# ---- ML адаптеры ----

def _base_meta(kwargs):
    return {'dataset_source': kwargs.get('dataset_source', 'data_utils'),
            'generation_params': kwargs.get('generation_params', {}),
            'source_path': kwargs.get('source_path', '')}

def run_lstm(series, seed, window_size=50, threshold=0.89, nms_min_distance=50,
             hidden_size=256, num_layers=2, dropout=0.2, weights_path=None, **kwargs):
    import random; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    series = np.asarray(series, dtype=float).reshape(-1)
    model = LSTMChangePointDetector(1, hidden_size, num_layers, dropout)
    if weights_path and Path(weights_path).exists():
        ckpt = torch.load(weights_path, map_location=DEVICE)
        model.load_state_dict(ckpt['state_dict'])
        model.to(DEVICE)
        print(f'LSTM: loaded {weights_path}', flush=True)
    else:
        raise RuntimeError(f'LSTM weights not found: {weights_path}')
    scores_raw, _ = predict_on_series(model, series, window_size, DEVICE, threshold)
    preds = nms_predictions(np.nan_to_num(scores_raw), threshold=threshold, min_distance=nms_min_distance)
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    meta = _base_meta(kwargs); meta['seed'] = int(seed)
    return np.where(preds == 1)[0].tolist(), scores_raw, meta

def run_gru(series, seed, window_size=50, threshold=0.99, nms_min_distance=50,
            hidden_size=128, num_layers=2, dropout=0.2, weights_path=None, **kwargs):
    import random; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    series = np.asarray(series, dtype=float).reshape(-1)
    model = GRUChangePointDetector(1, hidden_size, num_layers, dropout)
    if weights_path and Path(weights_path).exists():
        ckpt = torch.load(weights_path, map_location=DEVICE)
        model.load_state_dict(ckpt['state_dict'])
        model.to(DEVICE)
        print(f'GRU: loaded {weights_path}', flush=True)
    else:
        raise RuntimeError(f'GRU weights not found: {weights_path}')
    scores_raw, _ = predict_on_series(model, series, window_size, DEVICE, threshold)
    preds = nms_predictions(np.nan_to_num(scores_raw), threshold=threshold, min_distance=nms_min_distance)
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    meta = _base_meta(kwargs); meta['seed'] = int(seed)
    return np.where(preds == 1)[0].tolist(), scores_raw, meta

def run_transformer(series, seed, window_size=50, threshold=0.7, nms_min_distance=50,
                    d_model=256, nhead=8, num_layers=6, dim_feedforward=512, dropout=0.1,
                    weights_path=None, **kwargs):
    import random; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    series = np.asarray(series, dtype=float).reshape(-1)
    model = TransformerChangePointDetector(1, d_model, nhead, num_layers, dim_feedforward, dropout)
    if weights_path and Path(weights_path).exists():
        ckpt = torch.load(weights_path, map_location=DEVICE)
        model.load_state_dict(ckpt['state_dict'])
        model.to(DEVICE)
        print(f'Transformer: loaded {weights_path}', flush=True)
    else:
        raise RuntimeError(f'Transformer weights not found: {weights_path}')
    scores_raw, _ = predict_on_series(model, series, window_size, DEVICE, threshold)
    preds = nms_predictions(np.nan_to_num(scores_raw), threshold=threshold, min_distance=nms_min_distance)
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    meta = _base_meta(kwargs); meta['seed'] = int(seed)
    return np.where(preds == 1)[0].tolist(), scores_raw, meta

def run_catboost(series, seed, window_size=50, threshold=0.57, nms_min_distance=50,
                 weights_path=None, **kwargs):
    from catboost import CatBoostClassifier
    series = np.asarray(series, dtype=float).reshape(-1)
    if weights_path and Path(weights_path).exists():
        model = CatBoostClassifier()
        model.load_model(weights_path)
        print(f'CatBoost: loaded {weights_path}', flush=True)
    else:
        raise RuntimeError(f'CatBoost weights not found: {weights_path}')
    scores_raw, _ = predict_catboost(model, series, window_size, threshold)
    preds = nms_predictions(np.nan_to_num(scores_raw), threshold=threshold, min_distance=nms_min_distance)
    meta = _base_meta(kwargs); meta['seed'] = int(seed)
    return np.where(preds == 1)[0].tolist(), scores_raw, meta

print('adapters OK')


In [ ]:
def compute_auc(true_cps, scores, series_length, margin):
    if scores is None or not isinstance(scores, np.ndarray) or scores.size == 0:
        return {'roc_auc': None, 'pr_auc': None}
    scores_flat = scores.ravel()
    if len(scores_flat) != series_length:
        return {'roc_auc': None, 'pr_auc': None}
    valid_mask = np.isfinite(scores_flat)
    if valid_mask.sum() < 100:
        return {'roc_auc': None, 'pr_auc': None}
    labels = np.zeros(series_length, dtype=int)
    for cp in true_cps:
        lo = max(0, cp - margin)
        hi = min(series_length, cp + margin + 1)
        labels[lo:hi] = 1
    labels_valid = labels[valid_mask]
    scores_valid = scores_flat[valid_mask]
    if labels_valid.sum() == 0 or labels_valid.sum() == len(labels_valid):
        return {'roc_auc': None, 'pr_auc': None}
    try:
        roc = float(roc_auc_score(labels_valid, scores_valid))
        pr = float(average_precision_score(labels_valid, scores_valid))
    except ValueError:
        return {'roc_auc': None, 'pr_auc': None}
    return {'roc_auc': round(roc, 4), 'pr_auc': round(pr, 4)}


def compute_metrics(true_cps, pred_cps, scores, series_length, margin, nan_mae):
    if not true_cps and not pred_cps:
        return {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'mae': 0.0,
                'n_true': 0, 'n_pred': 0, 'roc_auc': None, 'pr_auc': None}
    pairs = []
    for i, t in enumerate(true_cps):
        for j, p in enumerate(pred_cps):
            d = abs(t - p)
            if d <= margin: pairs.append((i, j, float(d)))
    pairs.sort(key=lambda x: x[2])
    used_t, used_p, errors = set(), set(), []
    for i, j, d in pairs:
        if i not in used_t and j not in used_p:
            used_t.add(i); used_p.add(j); errors.append(d)
    tp = len(used_t)
    fp = len(pred_cps) - len(used_p)
    fn = len(true_cps) - len(used_t)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2.0 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    mae = float(np.mean(errors)) if errors else nan_mae
    result = {'precision': round(precision, 6), 'recall': round(recall, 6),
              'f1': round(f1, 6), 'mae': round(mae, 4),
              'n_true': len(true_cps), 'n_pred': len(pred_cps)}
    result.update(compute_auc(true_cps, scores, series_length, margin))
    return result


REGISTRY = {
    'Pettitt': pettitt,
    'SNHT': snht,
    'BuishandQ': buishand_q,
    'BuishandRange': buishand_range,
    'BuishandLR': buishand_likelihood_ratio,
    'BuishandU': buishand_u,
    'Chow': chow,
    'MDL': mdl,
    'SWAB': swab,
    'GP': gp,
    'Nyblom': nyblom,
    'CUSUM': run_cusum,
    'LSTM': run_lstm,
    'GRU': run_gru,
    'CatBoost': run_catboost,
    'LSTM_v2': run_lstm,
    'GRU_v2': run_gru,
    'Transformer_v2': run_transformer,
}

METHOD_PARAMS = {
    'Nyblom': {'m': 2000},
    'LSTM': {
        'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2,
        'window_size': 50, 'threshold': 0.89, 'nms_min_distance': 50,
        'weights_path': _find_model('lstm_h256_l2_seq5000_n200_all_noise_seed42.pt'),
    },
    'GRU': {
        'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2,
        'window_size': 50, 'threshold': 0.99, 'nms_min_distance': 50,
        'weights_path': _find_model('gru_h128_l2_seq5000_n100_all_noise_seed42.pt'),
    },
    'CatBoost': {
        'window_size': 50, 'threshold': 0.57, 'nms_min_distance': 50,
        'weights_path': _find_model('catboost_seq5000_n50_seed42.cbm'),
    },
    'LSTM_v2': {
        'hidden_size': 256, 'num_layers': 2, 'dropout': 0.2,
        'window_size': 100, 'threshold': 0.15, 'nms_min_distance': 10,
        'weights_path': _find_model('lstm_v2_h256_l2_w100_seq5000_n200_all_noise_seed42.pt'),
    },
    'GRU_v2': {
        'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2,
        'window_size': 100, 'threshold': 0.15, 'nms_min_distance': 10,
        'weights_path': _find_model('gru_v2_h128_l2_w100_seq5000_n100_all_noise_seed42.pt'),
    },
    'Transformer_v2': {
        'd_model': 256, 'nhead': 8, 'num_layers': 6, 'dim_feedforward': 512, 'dropout': 0.1,
        'window_size': 100, 'threshold': 0.15, 'nms_min_distance': 10,
        'weights_path': _find_model('transformer_v2_dm256_h8_l6_w100_seq5000_n100_all_noise_seed42.pt'),
    },
}

print('REGISTRY:', list(REGISTRY.keys()))
print('Models found:')
for name in ['LSTM', 'GRU', 'CatBoost', 'LSTM_v2', 'GRU_v2', 'Transformer_v2']:
    wp = METHOD_PARAMS.get(name, {}).get('weights_path')
    status = 'OK' if wp and Path(wp).exists() else 'MISSING'
    print(f'  {name:20s}: {status} ({wp})')

In [ ]:
all_summaries = {}

for scenario in SCENARIOS:
    D = scenario['D']
    noise_type = scenario['noise_type']
    tag = scenario['tag']
    GEN = {'length': 2000, 'dt': 1.0, 'D': D, 'noise_type': noise_type}
    OUTPUT_DIR = BASE_OUTPUT / f'eval_multi_{tag}'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f'\n{"="*70}')
    print(f'SCENARIO: {tag}  (D={D}, noise={noise_type})')
    print(f'{"="*70}')

    per_series_rows = []
    method_accum = {m: [] for m in REGISTRY}
    methods = list(REGISTRY.keys())

    for series_idx in range(N_SERIES):
        test_seed = TEST_SEED_START + series_idx
        x_values, cp_labels = generate_dataset(
            length=GEN['length'], dt=GEN['dt'], D=GEN['D'],
            noise_type=GEN['noise_type'], seed=test_seed,
        )
        true_cps = np.where(cp_labels == 1)[0].tolist()
        base_kwargs = {
            'dataset_source': 'data_utils',
            'generation_params': {**GEN, 'seed': test_seed},
            'source_path': 'data_utils.py',
        }
        print(f'[{series_idx+1}/{N_SERIES}] seed={test_seed}  true_cps={len(true_cps)}')

        for method_name in methods:
            fn = REGISTRY[method_name]
            call_kwargs = {**base_kwargs, **METHOD_PARAMS.get(method_name, {})}
            t_start = time.time()
            try:
                change_points, scores, _ = fn(x_values, test_seed, **call_kwargs)
            except Exception as exc:
                print(f'  {method_name:20s}  ERROR: {exc}')
                continue
            elapsed = time.time() - t_start
            metrics = compute_metrics(true_cps, change_points, scores, len(x_values), MARGIN, NAN_MAE)
            per_series_rows.append({
                'method': method_name, 'series_seed': test_seed,
                'n_pred': len(change_points), 'n_true': len(true_cps),
                'precision': metrics['precision'], 'recall': metrics['recall'],
                'f1': metrics['f1'], 'mae': metrics['mae'],
                'roc_auc': metrics.get('roc_auc'), 'pr_auc': metrics.get('pr_auc'),
                'time_s': round(elapsed, 2),
            })
            method_accum[method_name].append(metrics)
            auc_str = ''
            if metrics.get('roc_auc') is not None:
                auc_str = f'  ROC={metrics["roc_auc"]:.3f}  PR={metrics["pr_auc"]:.3f}'
            print(f'  {method_name:20s}  CP={len(change_points):3d}  P={metrics["precision"]:.3f}  R={metrics["recall"]:.3f}  F1={metrics["f1"]:.3f}{auc_str}  ({elapsed:.1f}s)')
        print()

    if per_series_rows:
        fields = ['method', 'series_seed', 'n_pred', 'n_true',
                  'precision', 'recall', 'f1', 'mae', 'roc_auc', 'pr_auc', 'time_s']
        with (OUTPUT_DIR / 'per_series.csv').open('w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=fields)
            w.writeheader(); w.writerows(per_series_rows)

    summary_rows = []
    for method_name in methods:
        rows = method_accum[method_name]
        if not rows: continue
        arr_f1 = np.array([r['f1'] for r in rows])
        arr_p = np.array([r['precision'] for r in rows])
        arr_r = np.array([r['recall'] for r in rows])
        arr_mae = np.array([r['mae'] for r in rows])
        valid_mae = arr_mae[arr_mae < NAN_MAE]
        roc_vals = [r.get('roc_auc') for r in rows if r.get('roc_auc') is not None]
        pr_vals = [r.get('pr_auc') for r in rows if r.get('pr_auc') is not None]
        mean_roc = round(float(np.mean(roc_vals)), 4) if roc_vals else None
        mean_pr = round(float(np.mean(pr_vals)), 4) if pr_vals else None
        summary_rows.append({
            'method': method_name, 'n_series': len(rows),
            'mean_precision': round(float(arr_p.mean()), 4),
            'mean_recall': round(float(arr_r.mean()), 4),
            'mean_f1': round(float(arr_f1.mean()), 4),
            'std_f1': round(float(arr_f1.std()), 4),
            'mean_mae': round(float(np.mean(valid_mae)) if len(valid_mae) > 0 else NAN_MAE, 2),
            'mean_roc_auc': mean_roc, 'mean_pr_auc': mean_pr,
        })

    if summary_rows:
        fields = ['method', 'n_series', 'mean_precision', 'mean_recall',
                  'mean_f1', 'std_f1', 'mean_mae', 'mean_roc_auc', 'mean_pr_auc']
        with (OUTPUT_DIR / 'summary.csv').open('w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=fields)
            w.writeheader(); w.writerows(summary_rows)

    all_summaries[tag] = summary_rows

    print(f'\n--- {tag} results ---')
    print(f'{"Method":<20} {"P":>6} {"R":>6} {"F1":>6} {"+-F1":>6} {"MAE":>9} {"ROC":>6} {"PR":>6}')
    print('-' * 75)
    for row in sorted(summary_rows, key=lambda r: -r['mean_f1']):
        roc_s = f'{row["mean_roc_auc"]:>6.3f}' if row['mean_roc_auc'] is not None else '   N/A'
        pr_s = f'{row["mean_pr_auc"]:>6.3f}' if row['mean_pr_auc'] is not None else '   N/A'
        print(f'{row["method"]:<20} {row["mean_precision"]:>6.3f} {row["mean_recall"]:>6.3f} {row["mean_f1"]:>6.3f} {row["std_f1"]:>6.3f} {row["mean_mae"]:>9.1f} {roc_s} {pr_s}')

print(f'\n\nDone! All {len(SCENARIOS)} scenarios complete.')

In [ ]:
print('\n\n=== CROSS-SCENARIO COMPARISON (F1) ===\n')
tags = [s['tag'] for s in SCENARIOS]
methods = list(REGISTRY.keys())

header = f'{"Method":<20}' + ''.join(f'{t:>12}' for t in tags)
print(header)
print('-' * len(header))

for method_name in methods:
    row = f'{method_name:<20}'
    for tag in tags:
        f1 = None
        for r in all_summaries.get(tag, []):
            if r['method'] == method_name:
                f1 = r['mean_f1']
                break
        row += f'{f1:>12.3f}' if f1 is not None else f'{"N/A":>12}'
    print(row)